In [20]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import torch
from transformers import pipeline
from tqdm import tqdm

In [21]:
# --- AYARLAR ---
# Eğittiğiniz modelin bulunduğu klasörün adı (Model.ipynb'deki OUTPUT_DIR)
MODEL_KLASORU = "xlm_roberta_model" 
YENI_VERI_CSV = "yeni_cekilen_yorumlar.csv"  # Kazıma ile elde ettiğiniz dosya
METIN_KOLONU = "Yorum Metni"                 # CSV'deki yorumların bulunduğu kolon adı
SONUC_CSV = "etiketlenmis_yeni_yorumlar.csv" # Çıktı dosyası
GRAFIK_KLASORU = "yeni_analiz_raporlari"

In [22]:
os.makedirs(GRAFIK_KLASORU, exist_ok=True)

# Etiket Haritası (Model.ipynb ile uyumlu)
ETIKET_ISIMLERI = {
    "LABEL_0": "olumlu",
    "LABEL_1": "olumsuz",
    "LABEL_2": "nötr",
    "LABEL_3": "öneri/tavsiye",
    "LABEL_4": "şikayet"
}

In [23]:
# Regex Kuralları
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
ZAMAN_RE = re.compile(r"\b\d{1,2}:\d{2}(:\d{2})?\b")

In [24]:
def metin_on_isle(metin):
    metin = str(metin)
    
    # 1. URL, Email ve Zaman Damgası Temizliği
    metin = URL_RE.sub("", metin)
    metin = EMAIL_RE.sub("", metin)
    metin = ZAMAN_RE.sub("", metin)
    
    # 2. @etiketleri standart formata çevirme
    metin = re.sub(r"@\w+", "<kullaniciadi>", metin)
    
    # 3. Gereksiz noktalama işaretlerini silme (Emojiler ve ! ? korunur)
    metin = re.sub(r'[\.\,\;\:\(\)\{\}\"\'\-\_\|\*\&\^\%\$\#\~\`\/\\]', ' ', metin)
    metin = re.sub(r" +", " ", metin).strip()
    
    # 4. Uzatılmış harf normalizasyonu ve Vurgu (Büyük harf)
    kelimeler = metin.split()
    yeni_kelimeler = []
    for kelime in kelimeler:
        if re.search(r"(.)\1{2,}", kelime):
            temiz_kelime = re.sub(r"(.)\1{2,}", r"\1\1", kelime)
            yeni_kelimeler.append(temiz_kelime.upper())
        else:
            yeni_kelimeler.append(kelime)
            
    return " ".join(yeni_kelimeler)

In [25]:
print("1. Veri seti yükleniyor...")
df = pd.read_csv("test.csv")  # Kazıma ile elde ettiğiniz dosya

# Sadece boş olmayan metinleri alalım
df = df[df[METIN_KOLONU].notna()].reset_index(drop=True)

print("2. Veri ön işleme uygulanıyor...")
df["temiz_metin"] = df[METIN_KOLONU].apply(metin_on_isle)

print(f"3. Model yükleniyor... ({MODEL_KLASORU})")
# Cihaz ayarı: GPU varsa 0, CPU ise -1
device = 0 if torch.cuda.is_available() else -1 
clf = pipeline("text-classification", model=MODEL_KLASORU, tokenizer=MODEL_KLASORU, device=device)

print("4. Yorumlar etiketleniyor (Bu işlem veri boyutuna göre sürebilir)...")
# Tahminleri al (Batch processing ile daha hızlı)
tahminler = clf(df["temiz_metin"].tolist(), batch_size=32, truncation=True, max_length=80)

# Çıkan sonuçları (LABEL_X) anlamlı kelimelere çevir ve Dataframe'e ekle
df["Tahmin_Etiketi"] = [ETIKET_ISIMLERI[t["label"]] for t in tahminler]
df["Tahmin_Skoru"] = [round(t["score"], 3) for t in tahminler]

# Sonuçları CSV olarak kaydet
df.to_csv(SONUC_CSV, index=False, encoding="utf-8-sig")
print(f"Etiketleme tamamlandı! Sonuçlar '{SONUC_CSV}' dosyasına kaydedildi.")

1. Veri seti yükleniyor...
2. Veri ön işleme uygulanıyor...
3. Model yükleniyor... (xlm_roberta_model)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

4. Yorumlar etiketleniyor (Bu işlem veri boyutuna göre sürebilir)...
Etiketleme tamamlandı! Sonuçlar 'etiketlenmis_yeni_yorumlar.csv' dosyasına kaydedildi.


In [ ]:
pd.set_option('display.max_colwidth', None)

In [51]:
df_sonuc = pd.read_csv(SONUC_CSV)
df_sonuc[["Yorum Metni","Tahmin_Etiketi","Tahmin_Skoru"]].head()

,Yorum Metni,Tahmin_Etiketi,Tahmin_Skoru
0,Evliya ve penguenin kendi seçimiydi peki ya wilson…,nötr,0.967
1,Ebrunun kalemi çok güçlü yazdığı skeçleri izlerken hiç sıkılmıyorum,olumlu,0.976
2,Evliyanın teklif metni çok güzeldi,olumlu,0.994
3,Beyinsiz seyircinin sürekli alkislamasi çok itici. Ulan zaten sahneye çıkmaları onların işi her biri geldiğinde iyi kötü alkışı gözünüze sokun,olumsuz,0.958
4,Uzun zaman sonra baya güldüm iyiydi,olumlu,0.994


In [28]:
print("5. Görselleştirme aşamasına geçiliyor...")

# --- GRAFİK 1: Sınıf Dağılımı Bar Grafiği ---
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="Tahmin_Etiketi", order=df["Tahmin_Etiketi"].value_counts().index, palette="viridis")
plt.title("Yeni Video: Yorum Duygu ve Niyet Dağılımı")
plt.xlabel("Kategoriler")
plt.ylabel("Yorum Sayısı")

# Barların üzerine sayıları yazdırma
for p in plt.gca().patches:
    plt.gca().annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                       ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(GRAFIK_KLASORU, "duygu_dagilimi.png"), dpi=150)
plt.close()
print(" -> 'duygu_dagilimi.png' kaydedildi.")

# --- GRAFİK 2: N-Gram Analizi ---
def ngram_frekans(metin_listesi, n=2, top_k=10):
    tum_ngramlar = []
    for metin in metin_listesi:
        # Görselleştirme için özel karakterleri temizle ve küçük harfe çevir
        kelimeler = re.sub(r'[^\w\s]', '', str(metin)).lower().split()
        if len(kelimeler) >= n:
            tum_ngramlar.extend([" ".join(kelimeler[i:i+n]) for i in range(len(kelimeler)-n+1)])
    return Counter(tum_ngramlar).most_common(top_k)

# Mevcut olan tüm kategoriler için N-gram çizmeye çalışalım
incelenecek_kategoriler = df["Tahmin_Etiketi"].unique()

for kategori in incelenecek_kategoriler:
    kategori_df = df[df["Tahmin_Etiketi"] == kategori]
    yorum_sayisi = len(kategori_df)
    
    # En az 2 yorum varsa grafik çizmeye çalış (N-gram için biraz kelime lazım)
    if yorum_sayisi >= 2: 
        bigramlar = ngram_frekans(kategori_df["temiz_metin"].tolist(), n=2, top_k=10)
        
        if bigramlar:
            kelime_gruplari, frekanslar = zip(*bigramlar)
            
            plt.figure(figsize=(10, 5))
            sns.barplot(x=list(frekanslar), y=list(kelime_gruplari), palette="magma")
            plt.title(f"[{kategori.upper()}] En Sık Kullanılan 2'li Kelimeler ({yorum_sayisi} Yorum)")
            plt.xlabel("Tekrar Sayısı")
            plt.ylabel("Kelime Grupları (Bigram)")
            plt.tight_layout()
            
            dosya_adi = kategori.replace("/", "_") # Dosya isminde hata çıkmaması için
            kayit_yolu = os.path.join(GRAFIK_KLASORU, f"ngram_{dosya_adi}.png")
            plt.savefig(kayit_yolu, dpi=150)
            plt.close()
            print(f" -> '{kategori}' kategorisi için N-gram kaydedildi ({yorum_sayisi} yorum üzerinden).")
        else:
            print(f" -> '{kategori}' kategorisinde N-gram oluşturacak yeterli uzunlukta kelime bulunamadı.")
    else:
        print(f" -> '{kategori}' kategorisinde sadece {yorum_sayisi} yorum var, grafik çizilmedi.")

print(f"\\nTüm işlemler tamamlandı! Grafikler '{GRAFIK_KLASORU}' klasöründe.")

5. Görselleştirme aşamasına geçiliyor...
 -> 'duygu_dagilimi.png' kaydedildi.


C:\Users\05hde\AppData\Local\Temp\ipykernel_24472\91288430.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x="Tahmin_Etiketi", order=df["Tahmin_Etiketi"].value_counts().index, palette="viridis")
C:\Users\05hde\AppData\Local\Temp\ipykernel_24472\91288430.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=list(frekanslar), y=list(kelime_gruplari), palette="magma")


 -> 'olumsuz' kategorisi için N-gram kaydedildi (72 yorum üzerinden).
 -> 'nötr' kategorisi için N-gram kaydedildi (50 yorum üzerinden).


C:\Users\05hde\AppData\Local\Temp\ipykernel_24472\91288430.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=list(frekanslar), y=list(kelime_gruplari), palette="magma")
C:\Users\05hde\AppData\Local\Temp\ipykernel_24472\91288430.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=list(frekanslar), y=list(kelime_gruplari), palette="magma")


 -> 'olumlu' kategorisi için N-gram kaydedildi (55 yorum üzerinden).


C:\Users\05hde\AppData\Local\Temp\ipykernel_24472\91288430.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=list(frekanslar), y=list(kelime_gruplari), palette="magma")


 -> 'öneri/tavsiye' kategorisi için N-gram kaydedildi (4 yorum üzerinden).
\nTüm işlemler tamamlandı! Grafikler 'yeni_analiz_raporlari' klasöründe.
